# Distribution Distances and Learned Latent Embeddings

**UMBB, Faculty of Sciences — Department of Computer Science**
Master 2 I2A, Semester 3 — Course: RFVI

*Bhattacharyya, Jensen–Shannon, Mahalanobis and Euclidean metrics; ResNet embeddings and latent projection (continuation of Lab3)*

## 1. Objectives

This lab studies how objects are compared once they have been turned into vectors or distributions. It complements Lab3: instead of letting an MLP learn a decision boundary, we explicitly choose a distance or a similarity and build the recognition around it.

The lab has two axes:

- **Distribution and matrix distances:** Bhattacharyya, Kullback–Leibler, Jensen–Shannon, Mahalanobis and the Euclidean baseline;
- **Learned representations:** a ResNet pretrained embedding, projected into a compact latent space by a variational autoencoder (VAE).

This lab builds directly on Lab3: the Flowers split (Train / Validation / Test) is reloaded from `data/split/` without being recomputed.

By the end of this lab, the student will be able to:

- represent an image as a distribution (color histogram) and as a vector (ResNet embedding);
- implement and contrast several distances and similarities, and study their symmetry, bounds and invariances;
- classify images by minimizing a distance to class-conditional models;
- project a CNN embedding into a latent space with a convolutional variational autoencoder;
- inspect the latent geometry (t-SNE / UMAP), interpolate in it and sample from the prior;
- build a content-based retrieval engine and measure its quality;
- compare hand-crafted distributions, the CNN embedding and the latent projection.

## 2. Context and Problem Statement

Chapter 3 of the course distinguishes two families of measures.

On one side, **distances** quantify how far apart two objects are; on the other, **similarities** quantify how close they are. Many classical distances reduce to a vector comparison (Euclidean, Mahalanobis, cosine), while others act on distributions (histograms) (Bhattacharyya, Kullback–Leibler, Jensen–Shannon).

A second theme of the chapter is the embedding / latent distinction:

$$\text{embedding}: x \to z \quad \text{(we describe the object)},$$

$$\text{latent}: z \to c \quad \text{(a compact code of the object)}.$$

In this lab we study both, and we ask the central question:

> **Does the choice of metric change which representation is "better"?**

The general pipeline is:

$$\underbrace{\text{Image} \to \text{CNN}}_{\text{representation}} \;\to\; \underbrace{\text{latent VAE}}_{\text{optional projection}} \;\to\; \underbrace{\text{distance / similarity}}_{\text{metric}} \;\to\; \underbrace{\text{class / retrieval}}_{\text{task}}$$

## 3. Dependencies and Setup

In [1]:
# Run once (remove the leading '#' / use '%pip' inside Jupyter)
# %pip install numpy pandas matplotlib opencv-python
# %pip install scikit-learn scikit-image
# %pip install torch torchvision umap-learn

> The ResNet weights are downloaded from `torch.hub` on the first run. An internet connection is required for that step; afterwards the weights are cached.

## 4. Part 1: Two Representations

Turn each image into a **distribution** and into a **vector**, using the same Flowers split as Lab3.

### 4.1 Reloading the split

In [5]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

SEED = 42

SPLIT_DIR = Path("../datasets/split/flowers")

CLASS_NAMES = [
    "daisy",
    "dandelion",
    "rose",
    "sunflower",
    "tulip"
]

NUM_CLASSES = len(CLASS_NAMES)   # 5

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

IMG_SIZE = (128, 128)

def load_image(path, size=IMG_SIZE):
    with Image.open(path) as img:
        img = img.convert("RGB")
        img = img.resize(size)
        return np.array(img)

def load_images(df):
    X = np.array([load_image(p) for p in df["filepath"]])
    y = df["label"].to_numpy()
    return X, y

X_train, y_train = load_images(train_df)
X_val,   y_val   = load_images(val_df)
X_test,  y_test  = load_images(test_df)

print("Train      :", len(X_train))
print("Validation :", len(X_val))
print("Test       :", len(X_test))

Train      : 3023
Validation : 648
Test       : 648


### 4.2 CNN embedding (ResNet)

We take a pretrained ResNet and cut off its final layer: the vector produced by global average pooling is our embedding $z$.

In [6]:

import torchvision
from torchvision.models import resnet18, ResNet18_Weights

net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
net.eval()

import torch
import torch.nn as nn

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.257, 0.280]

net.fc = nn.Identity()
net.eval()

def resnet_embed(image):
    # image: HWC RGB uint8, resized to 224x224
    x = torch.from_numpy(image)
    x = x.permute(2, 0, 1).float() / 255.0
    x = (x[2] - torch.tensor(MEAN)) / torch.tensor(STD)
    x = x.unsqueeze(0)
    with torch.no_grad():
        z = net(x)[0].numpy()
    return z

In [7]:
EMBED_SIZE = (224, 224)

def embed_images(X):
    z = np.array([
        resnet_embed(img)
        for img in X
    ])
    return z

z_train = embed_images(X_train)
z_val   = embed_images(X_val)
z_test  = embed_images(X_test)

print("z_train:", z_train.shape)

RuntimeError: The size of tensor a (128) must match the size of tensor b (3) at non-singleton dimension 2

> The embedding has a fixed dimension (resnet18 gives 512). **It must be verified by the program.**

### 4.3 Color histogram (distribution representation)

> **Note (added by the transcriber):** the handout describes the color histogram ($x \in \mathbb{R}^{48}$, non-negative, summing to one) but gives no code for it. The cell below is a minimal implementation consistent with that description (16 bins × 3 RGB channels, normalized to sum to one). Adapt it to your Lab3 version if it differs.

In [ ]:
N_BINS = 16  # 16 bins x 3 channels = 48 dimensions

def color_histogram(image, bins=N_BINS):
    # image: HWC RGB uint8 -> vector in R^(3*bins), non-negative, sums to 1
    h = np.concatenate([
        np.histogram(image[..., c], bins=bins, range=(0, 256))[0]
        for c in range(3)
    ]).astype(np.float64)
    return h / h.sum()

def hist_images(X):
    return np.array([color_histogram(img) for img in X])

h_train = hist_images(X_train)
h_val   = hist_images(X_val)
h_test  = hist_images(X_test)

print("h_train:", h_train.shape)   # (N_train, 48)

## 5. Part 2: Latent Projection with a Convolutional Variational Autoencoder

Learn a compact latent code $c$ directly from the original image $x$ using a small convolutional variational autoencoder, so that $x \approx \hat{x} = \mathrm{dec}(c)$ with $c \sim q_\phi(c \mid x)$.

In Part 1, the ResNet embedding $z$ describes the object in a semantic space learned for classification. Here we take a different route: we learn a compact visual code *directly from the pixels*, by asking a small variational autoencoder to reconstruct the image itself:

$$x \xrightarrow{\text{enc}} (\mu, \log\sigma^2) \xrightarrow{c = \mu + \sigma \odot \varepsilon} c \xrightarrow{\text{dec}} \hat{x} \approx x, \qquad c \in \mathbb{R}^{32},\; x \in [0,1]^{3\times128\times128}.$$

Unlike a plain autoencoder, the encoder outputs a probability distribution $q_\phi(c \mid x) = \mathcal{N}(\mu, \mathrm{diag}(\sigma^2))$ instead of a single deterministic point. The **reparametrization trick** $c = \mu + \sigma \odot \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, I)$ keeps the network differentiable, and a **KL term** regularizes the latent space so that it stays close to a standard Gaussian. This regularization is what makes interpolation and sampling in the latent space meaningful.

We keep the VAE deliberately simple (three conv layers, three transposed-conv layers): the goal is not to train a deep generative model, but to study what a compact probabilistic visual code encodes, how it behaves under interpolation, and how it compares to the ResNet embedding.

> The VAE is trained on **images**, not on the ResNet embedding. The two representations $z$ (ResNet) and $c$ (latent VAE) are therefore independent: $z$ captures semantics for classification, $c$ captures the visual content needed to reconstruct the image, under a Gaussian prior. Part 3 will let us compare the two.

### 5.1 Preparing the image tensors

Images are already loaded as `uint8` arrays of shape $128 \times 128 \times 3$ in Part 1. We convert them to `float32` tensors in $[0,1]$ with a channels-first layout, as expected by PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def images_to_tensor(X):
    '''X: (N, H, W, 3) uint8 -> (N, 3, H, W) float32 in [0,1].'''
    t = torch.from_numpy(X).float() / 255.0
    return t.permute(0, 3, 1, 2).contiguous()

x_tr = images_to_tensor(X_train)
x_va = images_to_tensor(X_val)
x_te = images_to_tensor(X_test)

print("x_tr:", x_tr.shape)   # (N, 3, 128, 128)

### 5.2 Network

The encoder compresses a $3 \times 128 \times 128$ image down to a 32-dimensional Gaussian in three stride-2 convolutions ($128 \to 64 \to 32 \to 16$, then global average pooling). Two linear heads produce the mean $\mu$ and the log-variance $\log\sigma^2$. The decoder mirrors this path with transposed convolutions and ends with a sigmoid to produce pixel values in $[0,1]$.

In [ ]:
class ConvVAE(nn.Module):

    def __init__(self, dim_c=32, img_size=128):
        super().__init__()

        assert img_size % 8 == 0, \
            "img_size must be a multiple of 8 (three stride-2 convs)."

        self.img_size = img_size
        self.base     = img_size // 8      # 16 for 128
        self.flat_dim = 128 * self.base * self.base
        self.dim_c    = dim_c

        # ---- Encoder (shared trunk) ----
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),         # img_size/2
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),         # img_size/4
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),         # img_size/8
            nn.AdaptiveAvgPool2d((1, 1)),  # 1 x 1
            nn.Flatten()
        )
        # Two heads: mean and log-variance
        self.fc_mu     = nn.Linear(128, dim_c)
        self.fc_logvar = nn.Linear(128, dim_c)

        # ---- Decoder ----
        self.fc = nn.Linear(dim_c, self.flat_dim)

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 128, kernel_size=4,
                               stride=2, padding=1),
            nn.ReLU(inplace=True),         # img_size/4
            nn.ConvTranspose2d(128, 64, kernel_size=4,
                               stride=2, padding=1),
            nn.ReLU(inplace=True),         # img_size/2
            nn.ConvTranspose2d(64, 32, kernel_size=4,
                               stride=2, padding=1),
            nn.ReLU(inplace=True),         # img_size
            nn.ConvTranspose2d(32, 3, kernel_size=4,
                               stride=2, padding=1),
            nn.Sigmoid()                   # img_size, values in [0,1]
        )

    def encode(self, x):
        h = self.enc(x)
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparametrize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, c):
        h = self.fc(c).view(-1, 128, self.base, self.base)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        c          = self.reparametrize(mu, logvar)
        xhat       = self.decode(c)
        return xhat, mu, logvar

The latent code used later (classification, retrieval, interpolation) is the **posterior mean $\mu$**, the natural deterministic embedding associated with each input.

### 5.3 Training

The VAE loss combines a reconstruction term (mean squared error between $x$ and $\hat{x}$) and a Kullback–Leibler divergence between the posterior $q_\phi(c \mid x)$ and the prior $\mathcal{N}(0, I)$:

$$\mathcal{L} = \underbrace{\|\hat{x} - x\|^2}_{\text{reconstruction}} + \beta\, \underbrace{D_{\mathrm{KL}}\big(\mathcal{N}(\mu, \sigma^2)\,\|\,\mathcal{N}(0, I)\big)}_{\text{regularization}},$$

with, in closed form for each latent dimension,

$$D_{\mathrm{KL}} = -\frac{1}{2} \sum_{j=1}^{d} \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right).$$

The hyperparameter $\beta$ weights the regularization; we use $\beta = 1$ (standard VAE) and mention $\beta \neq 1$ ($\beta$-VAE) as an extension.

In [ ]:
DIM_C = 32
BETA  = 1.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

vae = ConvVAE(dim_c=DIM_C, img_size=128).to(DEVICE)

optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

x_tr = x_tr.to(DEVICE)
x_va = x_va.to(DEVICE)

def vae_loss(x, xhat, mu, logvar, beta=BETA):
    rec = ((xhat - x) ** 2).mean()
    kld = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )
    return rec + beta * kld, rec, kld

n = len(x_tr)
batch_size = 64

for epoch in range(50):
    vae.train()
    perm = torch.randperm(n)
    for i in range(0, n, batch_size):
        idx = perm[i:i + batch_size]
        xb = x_tr[idx]

        xhat, mu, logvar = vae(xb)
        loss, rec, kld   = vae_loss(xb, xhat, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    vae.eval()
    with torch.no_grad():
        xhat_va, mu_va, lv_va = vae(x_va)
        val_loss, val_rec, val_kld = vae_loss(
            x_va, xhat_va, mu_va, lv_va
        )

    print(f"Epoch {epoch+1:03d} - "
          f"val_rec = {val_rec.item():.6f} "
          f"val_kld = {val_kld.item():.4f} "
          f"val_loss = {val_loss.item():.6f}")

Unlike the plain autoencoder, the VAE validation reconstruction error is typically a bit higher (the KL term prevents the model from collapsing to a near-identity mapping), but the latent space is now a proper probability distribution: samples of the prior decode to plausible flowers, and linear interpolation stays on the manifold. A well-trained $\beta = 1$ VAE on Flowers should reach a validation reconstruction MSE around $10^{-3}$–$10^{-2}$, with the KL term stabilizing (neither collapsing to 0 nor exploding); if the KL term vanishes, the model behaves like a plain autoencoder, and if it dominates, the reconstructions become blurry and class-agnostic.

### 5.4 Visualizing the reconstruction

Before using the latent codes, we check visually that the VAE actually reconstructs the flowers.

In [ ]:
import matplotlib.pyplot as plt

def show_reconstructions(vae, x_batch, n=6):
    vae.eval()
    with torch.no_grad():
        xhat, _, _ = vae(x_batch[:n])

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(x_batch[i].cpu().permute(1, 2, 0))
        axes[0, i].set_title("original")
        axes[0, i].axis("off")

        axes[1, i].imshow(xhat[i].cpu().permute(1, 2, 0))
        axes[1, i].set_title("reconstructed")
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()

show_reconstructions(vae, x_va)

### 5.5 Latent codes

In [ ]:
def to_latent(x_tensor, vae):
    '''Return the posterior mean mu as the deterministic latent code.'''
    vae.eval()
    with torch.no_grad():
        mu, _ = vae.encode(x_tensor.to(DEVICE))
    return mu.cpu().numpy()

c_train = to_latent(x_tr, vae)
c_val   = to_latent(x_va, vae)
c_test  = to_latent(images_to_tensor(X_test), vae)

print("c_train:", c_train.shape)   # (N_train, 32)
print("c_val  :", c_val.shape)
print("c_test :", c_test.shape)

The latent code has a small fixed dimension (32 in the default configuration). **It must be verified by the program.**

Unlike the ResNet embedding $z$, the latent code $c$ is learned **without any class label**: it is a purely unsupervised probabilistic visual compression. Because of the KL regularization, $c$ is approximately distributed as $\mathcal{N}(0, I)$, which Part 5 will exploit for interpolation and sampling. Part 4 will test whether this unsupervised code is enough to discriminate the five flower species.

## 6. Part 3: The Euclidean Distance

Implement the Euclidean distance and use it as the **unique metric** of this lab to compare vectors.

All representations of this lab produce a vector of fixed length: the color histogram ($x \in \mathbb{R}^{48}$, non-negative, summing to one), the ResNet embedding ($z \in \mathbb{R}^{512}$) and the latent code ($c \in \mathbb{R}^{32}$). To compare two objects, we use a single metric throughout: the Euclidean distance.

### 6.1 Definition

For two vectors $a, b \in \mathbb{R}^d$:

$$d_2(a, b) = \|a - b\|_2 = \sqrt{\sum_{i=1}^{d} (a_i - b_i)^2}.$$

Properties: symmetric, non-negative, zero iff $a = b$, and it satisfies the triangle inequality (it is a true metric).

In [ ]:
import numpy as np

def euclidean_distance(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(np.linalg.norm(a - b))

## 7. Part 4: Classification by Distance to Class Centroids

Fit a centroid per class on the training set, then classify every test sample by choosing the class whose centroid is the closest in Euclidean distance.

This is the classical **nearest-centroid** pattern recognizer. For each class $k$, we build the class mean $\mu_k$ from the training samples:

$$\mu_k = \frac{1}{|S_k|} \sum_{x \in S_k} x, \qquad S_k = \{x : y = k\}.$$

A test sample $x$ is assigned to:

$$\hat{y} = \arg\min_k \|x - \mu_k\|_2.$$

### 7.1 Fitting class models

In [ ]:
def build_class_models(X_tr, y_tr):
    models = {}
    for k in range(NUM_CLASSES):
        idx = (y_tr == k)
        models[k] = X_tr[idx].mean(axis=0)
    return models

### 7.2 Prediction

In [ ]:
def predict_by_distance(x, models):
    scores = {
        k: euclidean_distance(x, m)
        for k, m in models.items()
    }
    return min(scores, key=scores.get)


def accuracy_by_distance(X, y, models):
    correct = 0
    for x, t in zip(X, y):
        correct += (predict_by_distance(x, models) == t)
    return correct / len(X)

### 7.3 Applying the classifier

For each of the three representations, fit the class models on `X_train` and measure the accuracy on `X_val` and `X_test`.

| Representation   | Val accuracy | Test accuracy | Test F1 (macro) |
|------------------|--------------|---------------|-----------------|
| Color histogram  |              |               |                 |
| ResNet embedding |              |               |                 |
| Latent code (VAE)|              |               |                 |

Compute also the **macro F1-score** and display the **confusion matrix** of each representation.

In [ ]:
# TODO (student): fill the table of section 7.3.
# For each representation (color histogram, ResNet embedding, VAE latent):
#   1. fit the class models on the train features
#   2. compute val / test accuracy
#   3. compute the macro F1-score and display the confusion matrix
#
# from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

representations = {
    "Color histogram":   (h_train, h_val, h_test),
    "ResNet embedding":  (z_train, z_val, z_test),
    "Latent code (VAE)": (c_train, c_val, c_test),
}

### 7.4 Comparison with Lab3

On the same ResNet embedding, compare:

- the nearest-centroid classifier of this part;
- a k-Nearest Neighbors classifier with the same Euclidean metric;
- the MLP trained in Lab3 on the fused features.

| Method                      | Test accuracy |
|-----------------------------|---------------|
| Nearest-centroid (this lab) |               |
| k-NN (Euclidean)            |               |
| MLP (Lab3)                  |               |

In [ ]:
# TODO (student): fill the table of section 7.4.
# from sklearn.neighbors import KNeighborsClassifier

## 8. Part 5: Latent Geometry and Interpolation

Visualize the ResNet embedding and the VAE latent code in two dimensions, use the VAE to interpolate between two images in the latent space, and sample from the prior.

### 8.1 2D visualization

Project the latent codes to 2D with t-SNE (or UMAP) and color each point by its class label. Repeat for the ResNet embedding. The VAE latent should look smoother and closer to a single Gaussian blob, while the ResNet embedding may show sharper class clusters.

In [ ]:
from sklearn.manifold import TSNE

def plot_tsne(X, y, title):
    Z2 = TSNE(n_components=2, perplexity=30,
              random_state=42).fit_transform(X)
    plt.figure(figsize=(6, 5))
    for k in range(NUM_CLASSES):
        plt.scatter(Z2[y == k, 0], Z2[y == k, 1],
                    s=8, label=CLASS_NAMES[k])
    plt.title(title); plt.legend(); plt.tight_layout(); plt.show()

plot_tsne(c_val, y_val, "VAE latent (val)")
plot_tsne(z_val, y_val, "ResNet embedding (val)")

### 8.2 Interpolation in the latent space

Because the VAE prior is $\mathcal{N}(0, I)$, the latent space is smooth and interpolation is meaningful. Interpolating between the posterior means of two images yields a continuous sequence of plausible flowers.

In [ ]:
def interpolate(vae, x_a, x_b, steps=8):
    vae.eval()
    with torch.no_grad():
        mu_a, _ = vae.encode(x_a.unsqueeze(0).to(DEVICE))
        mu_b, _ = vae.encode(x_b.unsqueeze(0).to(DEVICE))
        alphas = torch.linspace(0, 1, steps, device=DEVICE)
        codes  = [(1 - a) * mu_a + a * mu_b for a in alphas]
        decs   = [vae.decode(c).squeeze(0).cpu() for c in codes]

    fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2))
    for i, d in enumerate(decs):
        axes[i].imshow(d.permute(1, 2, 0))
        axes[i].axis("off")
    plt.tight_layout(); plt.show()

interpolate(vae, x_va[0], x_va[1])

### 8.3 Sampling from the prior

Finally, draw $c \sim \mathcal{N}(0, I)$ and decode: a well-trained VAE should generate plausible (if blurry) flower-like images.

In [ ]:
def sample_prior(vae, n=8):
    vae.eval()
    with torch.no_grad():
        c    = torch.randn(n, DIM_C, device=DEVICE)
        imgs = vae.decode(c).cpu()
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2))
    for i in range(n):
        axes[i].imshow(imgs[i].permute(1, 2, 0))
        axes[i].axis("off")
    plt.tight_layout(); plt.show()

sample_prior(vae)